# 00 - Preparación de ambiente

Objetivo: preparar catálogo, esquemas, ubicaciones externas y parámetros base para el proyecto fintech medallion.

> La capa RAW se consume desde ADLS Gen2 usando `abfss://...` y Managed Identity. No se usa DBFS ni Volumes como capa RAW.


In [ ]:
# Databricks notebook source
# Widgets de configuración
dbutils.widgets.text("catalog_name", "fintech_lakehouse", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Schema Bronze")
dbutils.widgets.text("silver_schema", "silver", "Schema Silver")
dbutils.widgets.text("gold_schema", "gold", "Schema Gold")
dbutils.widgets.text("raw_base_path", "abfss://raw@<storage-account>.dfs.core.windows.net/fintech", "ADLS Raw Path")
dbutils.widgets.text("checkpoint_base_path", "abfss://checkpoints@<storage-account>.dfs.core.windows.net/fintech", "ADLS Checkpoints Path")

catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
raw_base_path = dbutils.widgets.get("raw_base_path")
checkpoint_base_path = dbutils.widgets.get("checkpoint_base_path")

print(f"Catalog: {catalog_name}")
print(f"RAW: {raw_base_path}")

## Configuración esperada de seguridad

La conexión a ADLS debe estar hecha mediante Managed Identity. En Azure Databricks normalmente se implementa con:

1. Azure Databricks Access Connector con Managed Identity.
2. Permisos RBAC sobre Storage Account: `Storage Blob Data Contributor` o permisos mínimos requeridos.
3. Unity Catalog Storage Credential.
4. External Location apuntando al contenedor RAW.

El siguiente bloque SQL es una plantilla. Ajusta nombres reales antes de ejecutar.


In [ ]:
# MAGIC %sql
# MAGIC -- Ejecutar con un usuario/admin que tenga permisos de Unity Catalog.
# MAGIC -- CREATE STORAGE CREDENTIAL fintech_mi_credential
# MAGIC -- WITH AZURE_MANAGED_IDENTITY '<access-connector-resource-id>';
# MAGIC
# MAGIC -- CREATE EXTERNAL LOCATION fintech_raw_location
# MAGIC -- URL 'abfss://raw@<storage-account>.dfs.core.windows.net/fintech'
# MAGIC -- WITH (STORAGE CREDENTIAL fintech_mi_credential);

# MAGIC -- GRANT READ FILES ON EXTERNAL LOCATION fintech_raw_location TO `data_engineers`;

In [ ]:
# Crear catálogo y schemas
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

spark.sql(f"USE CATALOG {catalog_name}")
print("Ambiente preparado correctamente")